# 04. Output Parser

LLM의 출력을 원하는 형식(문자열, JSON, 객체 등)으로 파싱합니다.

In [5]:
import os
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

True

In [6]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-5.5",
    temperature=0,
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ["LLM_BASE_URL"],
    use_responses_api=False,
)

## 1. StrOutputParser - 문자열 추출

`AIMessage` 객체에서 `.content` 문자열만 추출합니다.

In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}을 한 문장으로 설명해줘")
])

parser = StrOutputParser()

# prompt -> llm -> parser 순으로 수동 호출
messages = prompt.invoke({"topic": "LangChain"})
response = llm.invoke(messages)
result = parser.invoke(response)

print(type(result))  # str
print(result)

<class 'langchain_core.messages.base.TextAccessor'>
LangChain은 대규모 언어 모델(LLM)을 외부 데이터, 도구, 워크플로와 연결해 챗봇·에이전트 같은 AI 애플리케이션을 쉽게 만들 수 있게 해주는 프레임워크입니다.


## 2. JsonOutputParser - JSON 파싱

LLM이 JSON을 반환하도록 유도하고 딕셔너리로 파싱합니다.

In [8]:
from langchain_core.output_parsers import JsonOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("human", "파이썬 언어에 대한 정보를 name, year, creator 키를 가진 JSON으로 반환해줘. JSON만 반환해.")
])

parser = JsonOutputParser()

messages = prompt.invoke({}) # 변수 없으면 빈 딕셔너리로 넘기면 됨
response = llm.invoke(messages)
result = parser.invoke(response)

print(type(result))  # dict
print(result)

<class 'dict'>
{'name': 'Python', 'year': 1991, 'creator': 'Guido van Rossum'}


## 3. PydanticOutputParser - 구조체 파싱

Pydantic 모델을 정의하면 자동으로 포맷 지시사항을 생성하고 파싱합니다.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser # LLM이 돌려준 텍스트(JSON)를 Pydantic 객체로 바꿔 주는 파서
from pydantic import BaseModel, Field # pydantic 쓰는 이유는 validation check, type hinting, auto docstring 등등 다양한 기능을 제공하기 때문

# 원하는 데이터의 출력 구조 정의. LLM에게 받고 싶은 답변의 형태를 클래스로 설계.
class MovieReview(BaseModel):
    title: str = Field(description="영화 제목")
    score: int = Field(description="10점 만점 점수")
    summary: str = Field(description="한 줄 감상")

# 답변을 MovieReview 형태로 해석해라"라고 설정한 파서를 만드는 줄
parser = PydanticOutputParser(pydantic_object=MovieReview)
# 답변 전: LLM에게 줄 형식 지시문 만들기 (get_format_instructions())
# 답변 후: LLM의 JSON 텍스트를 MovieReview 객체로 바꾸고 타입 검사하기 (parser.invoke(response))

# 파서가 자동으로 포맷 지시사항 생성
# 사용 설명서
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "영화 제목", "title": "Title", "type": "string"}, "score": {"description": "10점 만점 점수", "title": "Score", "type": "integer"}, "summary": {"description": "한 줄 감상", "title": "Summary", "type": "string"}}, "required": ["title", "score", "summary"]}
```


In [10]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 영화 평론가입니다.\n{format_instructions}"),
    ("human",  "{movie} 영화를 리뷰해줘")
])

messages = prompt.invoke({
    "movie": "인터스텔라",
    "format_instructions": parser.get_format_instructions()
})
response = llm.invoke(messages)
result   = parser.invoke(response)

print(type(result))    # <class MovieReview>
print(result.title)    # 인터스텔라
print(result.score)    # 10
print(result.summary)


<class '__main__.MovieReview'>
인터스텔라
9
광활한 우주와 인간의 사랑을 과학적 상상력으로 엮어낸, 감정과 스케일 모두 압도적인 SF 걸작.


## 4. CommaSeparatedListOutputParser - 리스트 파싱

In [12]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("human", "파이썬 웹 개발 프레임워크 Top 5를 쉼표로 구분해서 나열해줘. 설명 없이 이름만.")
])

messages = prompt.invoke({})
response = llm.invoke(messages)
result = parser.invoke(response)

print(type(result))  # list
print(result)

<class 'list'>
['Django', 'Flask', 'FastAPI', 'Pyramid', 'Tornado']
